# 03 DETR × Pascal VOC 2007（Colab 実行用）

**問い:** 事前学習はどれだけ効くのか。DETR はなぜ NMS なしで動くのか？

- 条件 A: COCO 事前学習済み DETR 全体（`facebook/detr-resnet-50`）をファインチューニング
- 条件 B: バックボーン（ResNet-50）だけ ImageNet 事前学習、Transformer はランダム初期化

**手順:** ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選んでから、上から順に実行する。
1条件あたり概ね40分以内（5エポック＋評価）を想定。最後に `results/` を zip でダウンロードする。

> 1エポック目のログの `eta` を見て、40分を大きく超えそうなら `EPOCHS` や `MAX_SIZE` を下げ、**A と B を同じ設定で**実行し直すこと。

In [ ]:
!nvidia-smi

## 1. リポジトリを clone して依存ライブラリを入れる

In [ ]:
REPO_URL = "https://github.com/kimhuisong/summer-cv-practice.git"
BRANCH = "main"  # まだマージしていないブランチで試すときはブランチ名に変える

import os
if not os.path.exists("/content/summer-cv-practice"):
    !git clone -b {BRANCH} {REPO_URL} /content/summer-cv-practice
%cd /content/summer-cv-practice/03_detr_voc2007
!git log --oneline -1

In [ ]:
# transformers は v4 と v5 で DETR の実装が変わっているので、動作確認したバージョンに固定する
!pip install -q "transformers==5.18.0" timm torchmetrics pycocotools scipy
import torch, transformers, torchvision
print(torch.__version__, torchvision.__version__, transformers.__version__, torch.cuda.is_available())

## 2. ハンガリアンマッチングの自前実装とライブラリの照合（数十秒）

In [ ]:
!python hungarian_check.py

## 3. データ（VOC 2007 trainval / test、約900MB）のダウンロード

公式（Oxford）→ 旧ホスト名 → ミラー（pjreddie.com）の順に試し、md5 で検証する。どこから取れたかはログに出る。

In [ ]:
from dataset import download_voc2007
download_voc2007("data", "trainval")
download_voc2007("data", "test")

## 4. 学習と評価

A と B は同じ設定（エポック数・バッチサイズ・入力サイズ・学習率）で実行する。

In [ ]:
EPOCHS = 5
MAX_SIZE = 800
COMMON = f"--epochs {EPOCHS} --max_size {MAX_SIZE} --batch_size 8 --num_workers 2"

In [ ]:
# 条件 A: COCO 事前学習済み DETR 全体
!python train.py --condition A {COMMON}
!python evaluate.py --condition A --max_size {MAX_SIZE}

In [ ]:
# 条件 B: ImageNet バックボーンのみ（Transformer はランダム初期化）
!python train.py --condition B {COMMON}
!python evaluate.py --condition B --max_size {MAX_SIZE}

In [ ]:
# A と B の比較（学習曲線・表）
!python evaluate.py --compare

## 5. 結果を見る

In [ ]:
import json
from IPython.display import Image, Markdown, display
display(Markdown(open("results/comparison/summary.md", encoding="utf-8").read()))
display(Image("results/comparison/comparison_curves.png"))
for name in ["A_coco_pretrained", "B_imagenet_backbone"]:
    m = json.load(open(f"results/{name}/eval_metrics.json", encoding="utf-8"))
    print(name, "mAP@0.5 =", round(m["metrics_no_nms"]["map_50"], 4), " mAP@[.5:.95] =", round(m["metrics_no_nms"]["map"], 4))
    print("  +NMS:", m["metrics_with_nms"], " duplicates:", m["duplicate_stats"])
    display(Image(f"results/{name}/detections.png"))
    display(Image(f"results/{name}/cross_attention_0.png"))
    display(Image(f"results/{name}/query_specialization.png"))

## 6. results/ を zip にしてダウンロード

チェックポイント（`checkpoints/`、1条件あたり約170MB）は zip に含めない。必要なら Google Drive に別途コピーする。

In [ ]:
!zip -r -q results_03_detr.zip results -x "results/*/checkpoints/*" "results/*_quick/*" "results/comparison_quick/*"
!ls -lh results_03_detr.zip
from google.colab import files
files.download("results_03_detr.zip")

In [ ]:
# （任意）チェックポイントを Google Drive に保存する
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/summer-cv-practice/03_detr_voc2007
# !cp -r results /content/drive/MyDrive/summer-cv-practice/03_detr_voc2007/